## PHASE 1: EXPLORATION AND DATA CLEANING
This section covers loading the data, initial exploration, merging, and handling missing values.

In [52]:
# Import necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Set pandas to display all columns (prevents truncation in wide datasets)
pd.set_option("display.max_columns", None)

# Note: If any of these imports fail, you may need to install them using:
# pip install pandas , numpy , seaborn , or matplotlib

### =========================================
### 1. Exploratory Data Analysis (EDA) and Data Cleaning
### =========================================

In [53]:
df_flights = pd.read_csv("Customer Flight Activity.csv") # Load dataset

display(df_flights.head(2)) # Flights dataset exploration
print("=" * 100)

display(df_flights.tail(2))
print("=" * 100)

display(df_flights.sample(2))
print(f"The flights dataframe has {df_flights.shape[0]} rows and {df_flights.shape[1]} columns")

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
405622,999982,2018,12,0,0,0,0,0.0,0,0
405623,999986,2018,12,0,0,0,0,0.0,0,0


,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
298242,681593,2018,6,7,0,7,3101,310.0,0,0
192873,472225,2017,12,10,0,10,1040,104.0,0,0


The flights dataframe has 405624 rows and 10 columns


In [54]:
df_loyalty = pd.read_csv("Customer Loyalty History.csv")

display(df_loyalty.head(2)) # Loyalty dataset exploration
print("=" * 100)

display(df_loyalty.tail(2))
print("=" * 100)

display(df_loyalty.sample(2))
print(f"The loyalty dataframe has {df_loyalty.shape[0]} rows and {df_loyalty.shape[1]} columns")

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
16735,906428,Canada,Yukon,Whitehorse,Y2K 6R0,Male,Bachelor,-57297.0,Married,Star,10018.66,2018 Promotion,2018,4,NaN,NaN
16736,652627,Canada,Manitoba,Winnipeg,R2C 0M5,Female,Bachelor,75049.0,Married,Star,83325.38,Standard,2015,12,2016.0,8.0


,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
15144,316752,Canada,Alberta,Banff,T4V 1D4,Female,Bachelor,101933.0,Married,Star,8564.77,Standard,2016,1,NaN,NaN
11,144514,Canada,British Columbia,Dawson Creek,U5I 4F1,Female,Bachelor,49618.0,Married,Star,3864.78,Standard,2016,6,NaN,NaN


The loyalty dataframe has 16737 rows and 16 columns


In [55]:
# Detailed information for Flights
print("--- Flights Data Summary Table ---")
flights_summary = pd.DataFrame({
    'Data Type': df_flights.dtypes,
    'Non-Null Count': df_flights.count(),
    'Null Count': df_flights.isnull().sum(),
    'Null Percentage (%)': (df_flights.isnull().sum() / len(df_flights)) * 100
}).round(2)
display(flights_summary)

--- Flights Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
Loyalty Number,int64,405624,0,0.0
Year,int64,405624,0,0.0
Month,int64,405624,0,0.0
Flights Booked,int64,405624,0,0.0
Flights with Companions,int64,405624,0,0.0
Total Flights,int64,405624,0,0.0
Distance,int64,405624,0,0.0
Points Accumulated,float64,405624,0,0.0
Points Redeemed,int64,405624,0,0.0
Dollar Cost Points Redeemed,int64,405624,0,0.0


In [56]:
# Detailed information for Loyalty
print("--- Loyalty Data Summary Table ---")
loyalty_summary = pd.DataFrame({
    'Data Type': df_loyalty.dtypes,
    'Non-Null Count': df_loyalty.count(),
    'Null Count': df_loyalty.isnull().sum(),
    'Null Percentage (%)': (df_loyalty.isnull().sum() / len(df_loyalty)) * 100
}).round(2)
display(loyalty_summary)

--- Loyalty Data Summary Table ---


,Data Type,Non-Null Count,Null Count,Null Percentage (%)
Loyalty Number,int64,16737,0,0.00
Country,object,16737,0,0.00
Province,object,16737,0,0.00
City,object,16737,0,0.00
Postal Code,object,16737,0,0.00
Gender,object,16737,0,0.00
Education,object,16737,0,0.00
Salary,float64,12499,4238,25.32
Marital Status,object,16737,0,0.00
Loyalty Card,object,16737,0,0.00


### Observations & Cleaning Strategy

Based on the exploratory data analysis, the following technical strategies have been defined for data cleaning and merging:
 
### **Flights Dataset:**
 * **Identifiers:** Retain `loyalty_number` as `int64` to optimize memory and join performance. As a unique ID, statistical aggregations (mean, median) are not applicable.
 * **Data Integrity & Valid Zeros:** Although there are no null values, we must verify that `loyalty_number`, `year`, and `month` are strictly > 0 to rule out corrupt records. Zeros in flight metrics (e.g., flights booked) represent valid inactive months for users and must be strictly preserved to avoid distorting the temporal behavior.
 * **Temporal Engineering:** Combine the `year` and `month` integer columns into a unified `datetime` feature (`flight_date`) for proper time-series analysis.
 * **Metric Validation:** Confirm that `total_flights` exactly equals the sum of `flights_booked` and `flights_with_companions`.
 * **Type Consistency:** Retain `distance` as `int` as it provides sufficient precision. However, cast `points_redeemed` and `dollar_cost_points_redeemed` to `float` to maintain consistency with the `points_accumulated` format.

### **Loyalty Dataset:**
 * **Relational Key:** `loyalty_number` remains an `int64` and will serve as the primary key for joining both dataframes.
 * **Categorical Normalization:** String columns (`country`, `province`, `city`, `postal_code`, `gender`, `education`, `marital_status`, `loyalty_card`, `enrollment_type`) must be normalized (converted to lowercase and stripped of whitespaces) to prevent pseudo-duplicate categories caused by casing inconsistencies.
 * **Missing Data & Anomalies (Salary):**  Approximately 25% of `salary` data is missing. We must first identify and neutralize anomalies (e.g., negative salaries) before applying median imputation to avoid skewing the distribution.
 * **Temporal Features & Cancellations (MNAR Strategy):** Over 87% of cancellation data is null, indicating active memberships (Missing Not At Random). To extract this business value, we will create a boolean `is_active` feature. Then, we will generate a unified `cancellation_date` (leaving active users as `NaT`), and finally, drop the original, redundant cancellation year and month columns to optimize memory.


#### Global Schema Standardization (columns)

In [57]:
def standardize_columns(df): # Converts all column names to snake_case format for consistency
    original_cols = df.columns.tolist()
    df.columns = [col.lower().replace(" ", "_") for col in df.columns]
    print(f"Headers standardized: {len(original_cols)} columns processed.")
    return df

In [58]:
# Apply column standardization to both datasets to ensure a uniform schema
df_flights = standardize_columns(df_flights)
df_loyalty = standardize_columns(df_loyalty)

print("\n--- Standardized Flights Columns ---")
print(df_flights.columns.tolist())

print("\n--- Standardized Loyalty Columns ---")
print(df_loyalty.columns.tolist())

Headers standardized: 10 columns processed.
Headers standardized: 16 columns processed.

--- Standardized Flights Columns ---
['loyalty_number', 'year', 'month', 'flights_booked', 'flights_with_companions', 'total_flights', 'distance', 'points_accumulated', 'points_redeemed', 'dollar_cost_points_redeemed']

--- Standardized Loyalty Columns ---
['loyalty_number', 'country', 'province', 'city', 'postal_code', 'gender', 'education', 'salary', 'marital_status', 'loyalty_card', 'clv', 'enrollment_type', 'enrollment_year', 'enrollment_month', 'cancellation_year', 'cancellation_month']


#### Categorical Data Cleaning 


In [59]:
def normalize_strings(df, columns): # Normalizes categorical values: lowercase and whitespace removal.
    for col in columns:
        if col in df.columns:
            df[col] = df[col].str.lower().str.strip()
    print(f"String normalization applied to: {columns}")
    return df

In [60]:
def get_categorical_columns(df): # Identifies columns with 'object' data type.
    return df.select_dtypes(include=['object']).columns.tolist()

def normalize_strings(df, columns): # Normalizes categorical values: lowercase and whitespace removal.
    for col in columns:
        if col in df.columns:
            df[col] = df[col].str.lower().str.strip()
    print(f"String normalization applied to: {columns}")
    return df

In [61]:
string_cols_loyalty = get_categorical_columns(df_loyalty)

df_loyalty = normalize_strings(df_loyalty, string_cols_loyalty) # Apply string normalization.

print("\n--- Normalized Categorical Columns (Loyalty Sample) ---")
display(df_loyalty[string_cols_loyalty].head(2))
# Note: Currently applied only to the Loyalty dataset as Flights contains no object columns.

String normalization applied to: ['country', 'province', 'city', 'postal_code', 'gender', 'education', 'marital_status', 'loyalty_card', 'enrollment_type']

--- Normalized Categorical Columns (Loyalty Sample) ---


,country,province,city,postal_code,gender,education,marital_status,loyalty_card,enrollment_type
0,canada,ontario,toronto,m2z 4k1,female,bachelor,married,star,standard
1,canada,alberta,edmonton,t3g 6y6,male,college,divorced,star,standard


In [107]:
print("\n--- Normalized Categorical Columns (Loyalty Sample) ---")
# Professional visualization: Using Title Case for display purposes only
display(df_loyalty[string_cols_loyalty].sample(2).style.format(lambda x: x.title() if isinstance(x, str) else x))


--- Normalized Categorical Columns (Loyalty Sample) ---


,country,province,city,postal_code,gender,education,marital_status,loyalty_card,enrollment_type
10837,Canada,British Columbia,Victoria,V10 6T5,Male,High School Or Below,Single,Star,Standard
755,Canada,Quebec,Quebec City,G1B 3L5,Female,Bachelor,Married,Star,2018 Promotion


### Observations & Merging Strategy
* **Join Operation:** Perform a `LEFT JOIN` using the Flights dataset as the left table and Loyalty as the right table on `loyalty_number`. This preserves every transactional record while broadcasting the demographic attributes to each corresponding flight month.